## Ch10-03 -- A synthesis record, and what it is not

This notebook introduces a synthesis record over notebook 01's traceability graph and notebook 02's judgment ledger; after running it you can see, in one place, what has real bidirectional verification evidence, what has only one-sided evidence, what formal proof exists disconnected from any requirement, and what an accountable engineer would still have to decide before actually shipping this design.

State the distinction this whole notebook rests on before building anything: a completed traceability graph and a judgment ledger are not sign-off itself. Sign-off is a human, accountable act; a design's own engineer decides, given everything the graph and the ledger show, whether to proceed. This notebook can show the inputs to that decision, honestly and within its own stated scope, but it cannot make the decision for anyone, and it does not try to. The record built below stays `disposition = "pending"`, the same as every other record this tutorial has ever built (SA-7): this chapter does not get to be the exception.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch10-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


Before assembling anything, a real negative control fitting this notebook's own new record: does `validate_record()` reject a draft of it whose own `counterevidence` is empty? `counterevidence` is exactly the field AGENTS.md 1.6 calls load-bearing, and this check is real and mechanized, unlike a different rule discussed just below that no tool enforces.

In [2]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

draft_missing_counterevidence = ReviewRecord(
    identifier="AI-C10-DRAFT",
    kind="asserted_inference",
    claim="A draft of the synthesis record built below, missing its own counterevidence.",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="The same criteria the real record below states.",
    premises=["a premise"],
    rationale="Because the graph and the ledger say so.",
    counterevidence="",   # intentionally empty
    disposition="pending",
    record_kind="worked_example",
)
errors = validate_record(draft_missing_counterevidence)
assert len(errors) > 0, "Expected validation to fail on empty counterevidence"
print(f"Negative control ok: errors={errors}")


Negative control ok: errors=['counterevidence is empty']


`validate_record()` correctly rejects this: a real, mechanized floor actually catching something, the same check Chapter 9's own placeholder record exercised. A different rule this tutorial follows just as strictly has no such mechanized floor: AGENTS.md 1.6 forbids ever recording `disposition="accepted"` (SA-7), but reading `src/toaster/evidence.py`'s own `validate_record()` shows what it actually checks: `identifier`, `claim`, `rationale` and `counterevidence` non-empty, `record_kind != "actual_review"`, and at least one premise for an `asserted_inference`, never the `disposition` value itself. That gap is real and worth naming plainly (recorded in `decisions/next-passes.md`), but it is not demonstrated here by constructing the forbidden value: AGENTS.md's own rule against `disposition="accepted"` has no negative-control exception, so this point is made in prose only, never in code. Every record this notebook goes on to build keeps `disposition="pending"`, checked explicitly at the end, not merely asserted.

With that settled, the two real inputs this synthesis draws on: notebook 01's own coverage-and-orphan finding, recomputed directly below against the loaded model (each notebook in this tutorial loads and runs independently, so nothing here is assumed from memory), and a short summary of notebook 02's own three-record ledger, hand-verified against that notebook's own real, printed output rather than rebuilt a third time.

In [3]:
from toaster.query import ApiIndex, requirement_coverage, tied_to_any_requirement

idx = ApiIndex(model)
coverage = {c["requirement"]: c for c in requirement_coverage(model, idx)}

delivered_energy_bounded = idx.by_qn["ToasterDemo::deliveredEnergyBoundedBySupply"]
tied_to_a_requirement = tied_to_any_requirement(model, "ToasterDemo::deliveredEnergyBoundedBySupply", idx)

print(f"heatGenerationReq: {coverage['ToasterDemo::heatGenerationReq']}")
print(f"timely: {coverage['ToasterDemo::timely']}")
print(f"deliveredEnergyBoundedBySupply tied to any requirement: {tied_to_a_requirement}")

assert coverage["ToasterDemo::heatGenerationReq"]["covered"] is True
assert coverage["ToasterDemo::timely"]["covered"] is False
assert not tied_to_a_requirement

heatGenerationReq: {'requirement': 'ToasterDemo::heatGenerationReq', 'satisfied_by': ['ToasterDemo::rated'], 'failed_by': ['ToasterDemo::weak'], 'covered': True}
timely: {'requirement': 'ToasterDemo::timely', 'satisfied_by': [], 'failed_by': ['ToasterDemo::slow'], 'covered': False}
deliveredEnergyBoundedBySupply tied to any requirement: False


The same three findings notebook 01 established, reproduced here directly: `heatGenerationReq` has real, bidirectional verification evidence (`rated` satisfies it, `weak` fails it); `timely` has only a one-sided negative claim and an unbound verify objective, no positive claim at all; and `deliveredEnergyBoundedBySupply`, the model's own strongest formal proof, is tied to no requirement by either of the two narrow checks notebook 01 runs (satisfy-by-subject; a direct reference from within a requirement's own body). Notebook 02's own ledger is cited next by identifier and conclusion, not rebuilt a third time; its own fields were already verified verbatim there.

In [4]:
ledger_summary = {
    "AS-C06": {"kind": "asserted_solution", "engineering_conclusion": "undetermined"},
    "AS-C08": {"kind": "asserted_solution", "engineering_conclusion": "supported"},
    "AI-C06": {"kind": "asserted_inference", "engineering_conclusion": "undetermined"},
}
for identifier, summary in ledger_summary.items():
    print(f"{identifier}: {summary}")


AS-C06: {'kind': 'asserted_solution', 'engineering_conclusion': 'undetermined'}
AS-C08: {'kind': 'asserted_solution', 'engineering_conclusion': 'supported'}
AI-C06: {'kind': 'asserted_inference', 'engineering_conclusion': 'undetermined'}


Notebook 01's graph and notebook 02's ledger are now both in hand. The rest of this notebook builds one new record synthesizing them, following the judgment-record construction zone (`toaster-review-protocol`): name each group of fields, narrate what it is for, print it, then assemble.

What is being claimed, and about what: the synthesis itself, not any single requirement or record.

In [5]:
claim = (
    "Notebook 01's traceability graph and notebook 02's judgment ledger, taken "
    "together, give an accountable engineer a real, honestly-scoped basis for "
    "exercising sign-off judgment over this model as it stands: heatGenerationReq "
    "is bidirectionally verified, timely traces to a real functional intent with "
    "no positive verification at all, and the model's own strongest formal "
    "evidence (deliveredEnergyBoundedBySupply) is tied to no stated requirement. "
    "This record synthesizes those findings; it does not itself constitute "
    "sign-off, which remains a human, accountable act this tutorial can show the "
    "inputs to but not perform."
)
model_ref = "ToasterDemo"
print(claim)


Notebook 01's traceability graph and notebook 02's judgment ledger, taken together, give an accountable engineer a real, honestly-scoped basis for exercising sign-off judgment over this model as it stands: heatGenerationReq is bidirectionally verified, timely traces to a real functional intent with no positive verification at all, and the model's own strongest formal evidence (deliveredEnergyBoundedBySupply) is tied to no stated requirement. This record synthesizes those findings; it does not itself constitute sign-off, which remains a human, accountable act this tutorial can show the inputs to but not perform.


What standard this claim is checked against (appropriateness): confined to exactly what notebooks 01 and 02 established, nothing wider.

In [6]:
scope = (
    "The three chains notebook 01's own traceability graph built (heatGenerationReq "
    "and its bidirectional satisfy claims; timely and its one-sided negative claim "
    "plus its unbound verify objective; deliveredEnergyBoundedBySupply, proved by "
    "Z3 but tied to no requirement), and the three records notebook 02's own "
    "ledger reconstructed (AS-C06, AS-C08, AI-C06). Confined to what those two "
    "notebooks actually established; not a claim about any part of the model "
    "neither notebook touched."
)
criteria = (
    "For each traced element: is there real, bidirectional verification evidence "
    "(a positive AND a negative claim), only one-sided evidence, or none at all? "
    "Does a proven formal property connect back to a stated requirement, or stand "
    "unconnected (the inverse of a failure mode Douglas's own traceability concern "
    "names: an unjustified widget, a design element with no requirement behind "
    "it; here, evidence with no requirement in front of it)?"
)
print(scope)
print(criteria)

The three chains notebook 01's own traceability graph built (heatGenerationReq and its bidirectional satisfy claims; timely and its one-sided negative claim plus its unbound verify objective; deliveredEnergyBoundedBySupply, proved by Z3 but tied to no requirement), and the three records notebook 02's own ledger reconstructed (AS-C06, AS-C08, AI-C06). Confined to what those two notebooks actually established; not a claim about any part of the model neither notebook touched.
For each traced element: is there real, bidirectional verification evidence (a positive AND a negative claim), only one-sided evidence, or none at all? Does a proven formal property connect back to a stated requirement, or stand unconnected (the inverse of a failure mode Douglas's own traceability concern names: an unjustified widget, a design element with no requirement behind it; here, evidence with no requirement in front of it)?


What is being taken as given: the three ledger records, cited by identifier, and the coverage/orphan results notebook 01 established, both already independently verified above and in notebook 02, not re-argued here.

In [7]:
premises = [
    "notebook 01's traceability graph: heatGenerationReq covered=True "
    "(satisfied_by=['rated'], failed_by=['weak']); timely covered=False "
    "(satisfied_by=[], failed_by=['slow'], plus TimelyToastTest's own unbound "
    "verify objective); deliveredEnergyBoundedBySupply tied to no requirement "
    "under two narrow, explicitly-named checks (satisfy-by-subject; a direct "
    "subsets/redefines/references/referent reference from within a "
    "RequirementDefinition/RequirementUsage's own body, exact-type owner only) "
    "-- not a claim that no tie could exist by any conceivable mechanism.",
    "AS-C06 (asserted_solution, Chapter 6): ResistanceCoil selected over a "
    "combustion alternative; engineering_conclusion=undetermined, argued from a "
    "domain premise, not a trade study.",
    "AS-C08 (asserted_solution, Chapter 8): deliveredEnergyBoundedBySupply proved "
    "by Z3; engineering_conclusion=supported, but the proof is a hand-restated "
    "lemma, not a solver-checked reference to HeatGenerator's own "
    "efficiencyBounded/deliveredEnergy.",
    "AI-C06 (asserted_inference, Chapter 6): the level-2 heat-generation branch's "
    "own stopping judgment; engineering_conclusion=undetermined, energyIn not "
    "wired to any producer, HeatingAssembly not composed into any Toaster "
    "candidate.",
]
assumption_refs = [
    "Toaster::cycleTime remains a settable, underived attribute (unchanged since "
    "Chapter 2); this record does not assume it has been derived from anything."
]
for p in premises:
    print(p)
print()
print(assumption_refs)

notebook 01's traceability graph: heatGenerationReq covered=True (satisfied_by=['rated'], failed_by=['weak']); timely covered=False (satisfied_by=[], failed_by=['slow'], plus TimelyToastTest's own unbound verify objective); deliveredEnergyBoundedBySupply tied to no requirement under two narrow, explicitly-named checks (satisfy-by-subject; a direct subsets/redefines/references/referent reference from within a RequirementDefinition/RequirementUsage's own body, exact-type owner only) -- not a claim that no tie could exist by any conceivable mechanism.
AS-C06 (asserted_solution, Chapter 6): ResistanceCoil selected over a combustion alternative; engineering_conclusion=undetermined, argued from a domain premise, not a trade study.
AS-C08 (asserted_solution, Chapter 8): deliveredEnergyBoundedBySupply proved by Z3; engineering_conclusion=supported, but the proof is a hand-restated lemma, not a solver-checked reference to HeatGenerator's own efficiencyBounded/deliveredEnergy.
AI-C06 (asserted_i

What supports the claim, and how (sufficiency): the two notebooks' own outputs above, cited directly.

In [8]:
evidence_refs = [
    "notebook 01's own requirement_coverage() output and orphan-evidence search "
    "(this chapter, reproduced above)",
    "notebook 02's own three reconstructed ReviewRecords, each independently "
    "validated and diffed field-for-field against its real original (this "
    "chapter)",
]
rationale = (
    "heatGenerationReq is the strongest-traced requirement in this model: a real "
    "positive claim and a real negative claim both exist, though its own 600 W "
    "threshold is still a free-standing engineering figure, not derived from any "
    "stated measure of effectiveness (AC-C06, cited by AI-C06). timely is the "
    "weakest: a real functional intent with a real allocation and real physical "
    "candidates, but zero positive verification, since cycleTime is still not "
    "derived from anything. deliveredEnergyBoundedBySupply is this tutorial's "
    "single strongest piece of formal evidence, proved by Z3 for every value its "
    "unbound features admit, and the one furthest from any actual requirement "
    "under this tutorial's own two narrow, explicitly-named checks: it is never "
    "named as the subject of a real satisfy relationship, and no element owned "
    "by a requirement's own body (exact RequirementDefinition/RequirementUsage "
    "type only) subsets, redefines, references or bare-names it directly. This "
    "is a deliberate, bounded pair of checks, not a claim of completeness -- by "
    "design it does not chase a connection-end, a sibling dependency/allocate/"
    "metadata relationship, Concern/Viewpoint ownership, or a transitive chain "
    "through an unowned intermediate -- and it is the inverse of the "
    "'unjustified widget' pattern Douglas's own traceability concern names. That "
    "is a separate fact from a real limitation on the proof's own scope (AS-C08's "
    "own counterevidence already says so): the proof is of a hand-restated "
    "companion lemma, not a solver-checked link to HeatGenerator's own real "
    "elements. Taken together, this is a partial, honestly-scoped basis for "
    "sign-off, not a completed one."
)
print(rationale)

heatGenerationReq is the strongest-traced requirement in this model: a real positive claim and a real negative claim both exist, though its own 600 W threshold is still a free-standing engineering figure, not derived from any stated measure of effectiveness (AC-C06, cited by AI-C06). timely is the weakest: a real functional intent with a real allocation and real physical candidates, but zero positive verification, since cycleTime is still not derived from anything. deliveredEnergyBoundedBySupply is this tutorial's single strongest piece of formal evidence, proved by Z3 for every value its unbound features admit, and the one furthest from any actual requirement under this tutorial's own two narrow, explicitly-named checks: it is never named as the subject of a real satisfy relationship, and no element owned by a requirement's own body (exact RequirementDefinition/RequirementUsage type only) subsets, redefines, references or bare-names it directly. This is a deliberate, bounded pair of c

What could be wrong, and what is still open (trustworthiness): named plainly, not folded into a tidier-sounding conclusion.

In [9]:
counterevidence = (
    "No physical realization of the toaster as a whole has been checked "
    "end-to-end. ResistanceCoil's own mechanism selection rests on a domain "
    "premise, not a trade study (AS-C06). HeatGenerator::energyIn is still not "
    "wired to a producer, and HeatingAssembly is not composed into any Toaster "
    "candidate (AI-C06). heatGenerationReq's own 600 W threshold is not derived "
    "from any stated measure of effectiveness (AC-C06, cited by AI-C06). "
    "Toaster::cycleTime remains unset from anything (unchanged since Chapter 2), "
    "so timely's own coverage gap cannot be closed by more querying alone, only "
    "by new modeling work. deliveredEnergyBoundedBySupply's Z3 proof is of a "
    "hand-restated companion lemma, not a solver-checked reference to "
    "HeatGenerator's own efficiencyBounded/deliveredEnergy (DEFERRED.md D-030, "
    "D-031), so its formal strength does not transfer to any named requirement."
)
residual_uncertainties = (
    "Whether ResistanceCoil remains the right mechanism selection once a supply "
    "and a control policy are modeled together (AS-C06's own open question). "
    "Whether the restated deliveredEnergyBoundedBySupply lemma would still hold "
    "if HeatGenerator's own efficiencyBounded or deliveredEnergy were edited "
    "without also updating the restatement; nothing in this toolchain checks "
    "that automatically (AS-C08's own residual). Whether cycleTime should ever "
    "be derived, and from what, before timely can be positively checked at all, "
    "and, if it were, whether nominal would actually satisfy it, which no "
    "evidence in this model addresses either way. None of this is decided here: "
    "it is exactly the kind of judgment an accountable engineer, not this "
    "tutorial, has to exercise before actually shipping this design."
)
print(counterevidence)
print()
print(residual_uncertainties)


No physical realization of the toaster as a whole has been checked end-to-end. ResistanceCoil's own mechanism selection rests on a domain premise, not a trade study (AS-C06). HeatGenerator::energyIn is still not wired to a producer, and HeatingAssembly is not composed into any Toaster candidate (AI-C06). heatGenerationReq's own 600 W threshold is not derived from any stated measure of effectiveness (AC-C06, cited by AI-C06). Toaster::cycleTime remains unset from anything (unchanged since Chapter 2), so timely's own coverage gap cannot be closed by more querying alone, only by new modeling work. deliveredEnergyBoundedBySupply's Z3 proof is of a hand-restated companion lemma, not a solver-checked reference to HeatGenerator's own efficiencyBounded/deliveredEnergy (DEFERRED.md D-030, D-031), so its formal strength does not transfer to any named requirement.

Whether ResistanceCoil remains the right mechanism selection once a supply and a control policy are modeled together (AS-C06's own op

Assembling the record from the named parts above. `kind="asserted_inference"` is the deliberate choice here, not `asserted_solution`: this record's own premises are literally child claims (the ledger's three records, plus notebook 01's own coverage and orphan findings) supporting a parent synthesis, exactly Hawkins' own "child claims supporting a parent" idea (SS3.1), not new evidence of its own.

In [10]:
signoff = ReviewRecord(
    identifier="AI-C10",
    kind="asserted_inference",
    claim=claim,
    model_ref=model_ref,
    content_hash=hash_content(source),
    scope=scope,
    criteria=criteria,
    premises=premises,
    assumption_refs=assumption_refs,
    evidence_refs=evidence_refs,
    rationale=rationale,
    counterevidence=counterevidence,
    residual_uncertainties=residual_uncertainties,
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(signoff)
print(f"Validation errors: {errors}")
print(f"identifier={signoff.identifier!r}  kind={signoff.kind!r}")
print(f"disposition={signoff.disposition!r}  record_kind={signoff.record_kind!r}")
print(f"engineering_conclusion={signoff.engineering_conclusion!r}")
assert errors == []
assert signoff.disposition == "pending"
assert signoff.record_kind == "worked_example"
conn.close()


Validation errors: []
identifier='AI-C10'  kind='asserted_inference'
disposition='pending'  record_kind='worked_example'
engineering_conclusion='undetermined'


`engineering_conclusion` stays `undetermined`, not `supported`: one requirement is genuinely covered, one is not, and the model's own strongest proof is tied to neither, so no single word honestly describes the model as a whole except the word that admits the mixture. Say plainly what this record is, and is not: it is a synthesis of two real, already-verified inputs into one honest, bounded statement of what is established, what is not, and what residual judgment remains. It is not sign-off. A completed traceability graph and judgment ledger tell an engineer what they are working with; deciding whether that is enough to actually proceed, given the real residual uncertainties named above, is a human, accountable act this tutorial can show the inputs to but cannot perform on the learner's behalf.

AGENTS.md's own account of emergence names what a real sign-off actually has to weigh, beyond anything this record states: "strong emergence (unanticipated; seen only in integration, test or operation) belongs to no layer. It is what sign-off judges." This chapter's own synthesis, however honestly built and however carefully bounded, cannot anticipate strong emergence by construction: it is a map of what has already been checked and what has not, not a forecast of what integration or operation might still reveal. It is exactly the kind of artifact a real engineer would weigh against that residual, unanticipated risk before deciding to proceed, not a substitute for weighing it. A traceable, honestly-scoped case makes that judgment easier to exercise well; it does not make the judgment itself unnecessary.

Three notebooks, one arc: a real traceability graph that traces two requirements very differently and finds one genuine orphaned proof; a judgment ledger over three real records, two of them still undetermined; and a synthesis record that states honestly what all of that does, and does not, establish, with `disposition="pending"` like every other record this tutorial has built.

Try the chapter exercise in `exercises/ch10/exercise.ipynb`: it asks you to build this same traceability graph, judgment ledger and sign-off synthesis over your own coffee-maker model.